# 08. 침묵 변이 처리 3가지 비교 (포함 / 제외 / 분리)

## 배경
- Day 2 목표: 침묵 변이(Silent) 처리 방식 3가지 비교
- 07에서 ① 포함 vs ② 제외는 비교함 → 포함이 CV +0.007~0.011 (단, 변이 개수 피처 stats와 함께 비교)
- ③ 분리(침묵 변이를 별도 행렬로)는 아직 검증하지 않음
- 변이 유형 함수를 `src/features.py`의 `mutation_features()`로 옮김 (01 EDA의 `mutation_type()` 규칙 사용)

## 가설
- H9: 침묵 변이를 별도 행렬로 분리하면(③) 포함(①)이나 제외(②)보다 CV가 오른다
  - 근거: 같은 유전자라도 침묵 변이와 기능 변이가 암종 구분에 다른 의미를 가질 수 있음
- H0: 분리해도 CV는 포함(①)보다 좋아지지 않는다

## 검증 방법
- 모델 고정: LR(class_weight='balanced'), C ∈ {0.05, 0.1}, 공통 CV
- **stats(변이 개수 피처)는 넣지 않음**: 07에서 CV만 올리고 LB에는 효과가 없었음
- 비교 조합
  | 조합 | 구성 | 의미 |
  | :--- | :--- | :--- |
  | S1 | any | ① 포함 (05와 같은 조건) |
  | S2 | nonsilent | ② 제외 |
  | S3 | nonsilent + silent | ③ 분리 |
  | S1t | any + trunc | ① + 절단 변이 (07의 F와 같은 구성) |
  | S3t | nonsilent + silent + trunc | ③ + 절단 변이 |
- 판단 기준: 같은 조건(trunc 유무)끼리 비교해 +0.01 이상이면 채택
- CV는 1차 필터. LB 제출 여부는 결과를 보고 따로 결정
- 모든 피처는 행 단위 계산 → Data Leakage 아님

In [1]:
import sys; sys.path.append('..')
import time, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features, MUT_TYPES
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)

t = time.time()
F_tr = mutation_features(train, genes)   # dict: any, nonsilent, silent, trunc, stats
F_te = mutation_features(test, genes)
print(f'{time.time()-t:.1f}s', {k: v.shape for k, v in F_tr.items()})

3.5s {'any': (6201, 4384), 'nonsilent': (6201, 4384), 'silent': (6201, 4384), 'trunc': (6201, 4384), 'stats': (6201, 9)}


#### 변이 유형 분포 확인 (01 EDA와 같은지)
- EDA: Missense 64.6% · Silent 26.3% · Nonsense 5.2% · Frameshift 3.9%

In [2]:
cnt = np.expm1(F_tr['stats'][[f'n_{t}_log' for t in MUT_TYPES]]).sum()
cnt.index = MUT_TYPES
print((cnt / cnt.sum() * 100).round(2))

# 침묵 변이만 있는 칸 / 침묵 + 기능 변이가 함께 있는 칸
only_silent = ((F_tr['silent'].values == 1) & (F_tr['nonsilent'].values == 0)).sum()
both = ((F_tr['silent'].values == 1) & (F_tr['nonsilent'].values == 1)).sum()
print(f'변이 칸 {F_tr["any"].values.sum():,}개 중 침묵만 {only_silent:,}개, 침묵+기능 함께 {both:,}개')

silent           26.280001
missense         64.610001
nonsense          5.230000
frameshift        3.880000
inframe_indel     0.000000
stop_loss         0.000000
other             0.000000
dtype: float32
변이 칸 218,893개 중 침묵만 53,020개, 침묵+기능 함께 8,613개


#### S1~S3t 비교

In [3]:
def make_lr(C):
    # fold마다 새 모델 생성
    return lambda: LogisticRegression(C=C, max_iter=3000, class_weight='balanced', random_state=SEED)

def build(F, parts):
    return pd.concat([F[p] for p in parts], axis=1)

combos = {
    'S1_any':                 ['any'],
    'S2_nonsilent':           ['nonsilent'],
    'S3_ns+silent':           ['nonsilent', 'silent'],
    'S1t_any+trunc':          ['any', 'trunc'],
    'S3t_ns+silent+trunc':    ['nonsilent', 'silent', 'trunc'],
}

rows = []
for name, parts in combos.items():
    X = build(F_tr, parts)
    for C in [0.05, 0.1]:
        t = time.time()
        r = run_cv(make_lr(C), X, y, verbose=False)
        rows.append(dict(set=name, C=C, n_feat=X.shape[1], cv=r['cv_mean'], std=r['cv_std'],
                         oof=r['oof_score'], sec=round(time.time() - t)))
        print(rows[-1])
res8 = pd.DataFrame(rows)
res8.pivot(index='set', columns='C', values='cv').loc[list(combos)].round(4)

{'set': 'S1_any', 'C': 0.05, 'n_feat': 4384, 'cv': 0.3838190233315374, 'std': 0.007927521435212194, 'oof': 0.3850511311538694, 'sec': 5}
{'set': 'S1_any', 'C': 0.1, 'n_feat': 4384, 'cv': 0.3817687059244309, 'std': 0.00644502067945055, 'oof': 0.3830000218575727, 'sec': 5}
{'set': 'S2_nonsilent', 'C': 0.05, 'n_feat': 4384, 'cv': 0.3576618256708349, 'std': 0.006889616997691679, 'oof': 0.35933912474646124, 'sec': 4}
{'set': 'S2_nonsilent', 'C': 0.1, 'n_feat': 4384, 'cv': 0.3594944169608324, 'std': 0.004943023090982429, 'oof': 0.36071137774105266, 'sec': 4}
{'set': 'S3_ns+silent', 'C': 0.05, 'n_feat': 8768, 'cv': 0.3760265206201836, 'std': 0.004797705086146993, 'oof': 0.3778934623365456, 'sec': 7}
{'set': 'S3_ns+silent', 'C': 0.1, 'n_feat': 8768, 'cv': 0.37013180478607655, 'std': 0.005402650510803303, 'oof': 0.3720370926697615, 'sec': 8}
{'set': 'S1t_any+trunc', 'C': 0.05, 'n_feat': 8768, 'cv': 0.3860342670112204, 'std': 0.008490607979218644, 'oof': 0.3876626954044262, 'sec': 8}
{'set': 'S1

C,0.05,0.10
set,,
S1_any,0.3838,0.3818
S2_nonsilent,0.3577,0.3595
S3_ns+silent,0.3760,0.3701
S1t_any+trunc,0.3860,0.3920
S3t_ns+silent+trunc,0.3863,0.3834


## 결과

**변이 유형 분포**: silent 26.28%, missense 64.61%, nonsense 5.23%, frameshift 3.88% → 01 EDA와 일치
- 변이 칸 218,893개 중 침묵 변이만 있는 칸 53,020개(24%), 침묵 + 기능 변이가 함께 있는 칸 8,613개

**CV 비교** (공통 CV, LR class_weight='balanced')

| 조합 | 구성 | 피처 수 | C=0.05 | C=0.1 | 최고 CV (± std) | 비교 대상 대비 |
| :--- | :--- | --: | :-: | :-: | :-: | :-: |
| S1 | any (① 포함) | 4,384 | **0.3838** | 0.3818 | 0.3838 ± 0.0079 | — |
| S2 | nonsilent (② 제외) | 4,384 | 0.3577 | 0.3595 | 0.3595 ± 0.0049 | S1 대비 **−0.024** |
| S3 | nonsilent + silent (③ 분리) | 8,768 | 0.3760 | 0.3701 | 0.3760 ± 0.0048 | S1 대비 **−0.008** |
| S1t | any + trunc | 8,768 | 0.3860 | **0.3920** | 0.3920 ± 0.0075 | — |
| S3t | nonsilent + silent + trunc | 13,152 | 0.3863 | 0.3834 | 0.3863 ± 0.0039 | S1t 대비 **−0.006** |

- S1(C=0.05) 0.3838은 05·06과 동일, S1t(C=0.1) 0.3920은 07 F(0.3921)와 사실상 동일
  → 유형 분류 규칙을 EDA 기준으로 바꾼 영향은 무시할 수준

## 결론
- **H9 기각**: 침묵 변이를 분리(③)해도 포함(①)보다 낮음 (trunc 없이 −0.008, trunc 포함 −0.006)
- 순위: **① 포함 > ③ 분리 > ② 제외** (trunc 유무와 관계없이 같은 순서)
- ② 제외는 −0.024로 크게 손해. 07(stats 포함 비교)의 −0.007보다 하락폭이 큼
  → 07에서는 stats의 침묵 변이 개수(n_silent)가 제외된 정보를 일부 보완하고 있었던 것으로 보임
- ③ 분리가 ① 포함보다 낮은 이유(추정)
  - 침묵 변이 정보는 "유전자에 변이가 있다"는 형태로 합쳐 둘 때 더 유용함
  - 나누면 피처가 2배가 되고 각 피처가 더 희소해져, 같은 규제(C)에서 신호가 약해짐
  - 단, ③은 std가 가장 작음(0.0039~0.0048) → 안정적이지만 점수는 낮음
- **침묵 변이 처리 방식은 ① 포함(any)을 팀 기본값으로 확정**
- trunc 행렬은 CV +0.008(S1 → S1t)이지만 07 F 제출에서 LB 효과가 없었음(0.2700) → 기본값에는 넣지 않고 선택 사항으로 둠
- 이번 실험은 LB에 영향을 줄 변경이 없으므로 제출하지 않음 (최고 LB는 05의 0.2739 유지)

## 다음
- 공통 피처: `src.features.mutation_features()` 사용, 침묵 변이는 `any`(포함)로 처리
- H7: 변이 0개 행 대응 → THYM 과예측 감소 (train OOF로 검증)
- H8: 다른 모델 계열(LightGBM 등)의 LB 확인 → LR 계열이 LB 0.27 근처에서 막혀 있으므로 우선순위 높음